# GeneMiner on Google Colab (Free GPU)

This notebook starts your existing FastAPI backend in Colab and exposes it with ngrok.
It is intended for research/dev, not for always-on production serving.


In [ ]:
import os
import subprocess
import platform
from pathlib import Path

print("Python:", platform.python_version())
print("Platform:", platform.platform())

REPO_URL = 'https://github.com/YOUR_USERNAME/GeneMiner-DKD.git'
REPO_DIR = Path('/content/GeneMiner-DKD')
print("Repo URL:", REPO_URL)
print("Repo Dir:", REPO_DIR)


## 1) Clone or update repo


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

os.chdir(REPO_DIR)


## 2) Runtime check
Enable GPU in Colab runtime for faster training, fallback to CPU is automatic.


In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('torch_device:', torch.device('cuda'))
else:
    print('No CUDA. Will run on CPU.')
    print('torch_device:', torch.device('cpu'))


## 3) Install dependencies
Install package dependencies and ngrok helper.


In [ ]:
import sys

os.chdir(REPO_DIR)

MIN_PYTHON_VERSION = (3, 13)


def run_cmd(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    return p.returncode == 0


colab_fallback_deps = [
    'torch>=2.0',
    'transformers>=4.36',
    'datasets>=2.16',
    'numpy',
    'pandas',
    'scikit-learn>=1.3',
    'requests',
    'mygene>=3.2',
    'accelerate>=0.26',
    'fastapi>=0.109',
    'uvicorn[standard]>=0.27',
    'python-multipart>=0.0.6',
    'pydantic-settings>=2.1',
    'openpyxl>=3.1',
    'biopython>=1.83',
]

if sys.version_info < MIN_PYTHON_VERSION:
    print('Python < 3.13 detected. Using Colab-compatible dependency path: requirements + fallback list.')
    if not run_cmd([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'backend/requirements-api.txt')]):
        print('requirements-api install failed. Falling back to explicit dependency list.')
    if not run_cmd([sys.executable, '-m', 'pip', 'install', *colab_fallback_deps]):
        raise RuntimeError('Dependency install failed. Please inspect pip logs above.')
else:
    print('Installing GeneMiner API dependencies...')
    install_ok = run_cmd([sys.executable, '-m', 'pip', 'install', '-e', '.[api]'])
    if not install_ok:
        print('Primary install failed. Retrying with --ignore-requires-python (Colab compatibility fallback)...')
        install_ok = run_cmd([sys.executable, '-m', 'pip', 'install', '-e', '.[api]', '--ignore-requires-python'])

    if not install_ok:
        print('Editable install still failed. Installing dependency fallback list...')
        if not run_cmd([sys.executable, '-m', 'pip', 'install', *colab_fallback_deps]):
            raise RuntimeError('Dependency install failed. Please inspect pip logs above.')

print('Installing pyngrok...')
if not run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok']):
    raise RuntimeError('pyngrok install failed. Please inspect pip logs above.')

print('Dependencies installation step completed.')


## 4) Prepare import paths and validate app import


In [ ]:
from pathlib import Path
import sys

BACKEND_DIR = REPO_DIR / 'backend'
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

os.environ['PYTHONPATH'] = f"{BACKEND_DIR}:{os.environ.get('PYTHONPATH', '')}"
os.environ.setdefault('CORS_ORIGINS', '[\"http://localhost:5173\", \"http://127.0.0.1:5173\"]')

from app.main import app
print('App title:', app.title)


## 5) Start FastAPI in background


In [ ]:
import subprocess
import time
import requests

os.chdir(BACKEND_DIR)
server_proc = subprocess.Popen([
    'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'
])
print('Backend PID:', server_proc.pid)
time.sleep(4)
try:
    r = requests.get('http://127.0.0.1:8000/health', timeout=20)
    print('Health:', r.status_code, r.text)
except Exception as e:
    print('Health check failed:', e)


## 6) Expose public URL by ngrok
Use an ngrok token for long sessions.


In [ ]:
from pyngrok import ngrok

# Optional: set token if required
# from getpass import getpass
# NGROK_TOKEN = getpass('Paste ngrok token: ')
# ngrok config add-authtoken <TOKEN>

public_tunnel = ngrok.connect(8000)
print('Public API URL:', public_tunnel.public_url)
print('Health check:', f'{public_tunnel.public_url}/health')


## 7) Use with React frontend
Set API base in your UI to the public URL shown above.


In [ ]:
BASE_URL = 'https://<your-ngrok-domain>'  # replace this
if '<your-ngrok-domain>' in BASE_URL:
    print('Replace BASE_URL with public_tunnel.public_url value.')
else:
    r = requests.get(f'{BASE_URL}/health', timeout=20)
    print(r.status_code, r.text)


## 8) Optional smoke check for PubMed/LitSuggest

Run this after you get `BASE_URL` and your backend is reachable.
This verifies:
- health endpoint
- project creation
- LitSuggest comparison endpoint
- optional PubMed Entrez fetch endpoint (requires valid email)


In [ ]:
import requests

base = BASE_URL.rstrip("/")

if "<your-ngrok-domain>" in base:
    raise ValueError("Replace BASE_URL with your ngrok public URL before running smoke check")

h = requests.get(f"{base}/health", timeout=20)
print("/health", h.status_code, h.text)

pr = requests.post(
    f"{base}/projects",
    json={"name":"Colab smoke check", "disease_key":"colab", "description":"automated"},
    timeout=20,
)
print("create project", pr.status_code, pr.text[:220])
if pr.status_code != 200:
    raise RuntimeError("Project create failed")
project_id = pr.json()["id"]

cmp = requests.post(
    f"{base}/projects/{project_id}/data/compare/litsuggest",
    json={
        "primary": [
            {"pmid": "10", "label": 1},
            {"pmid": "11", "label": 0},
        ],
        "litsuggest": [
            {"pmid": "10", "score": 0.87},
            {"pmid": "11", "score": 0.15},
        ],
        "score_threshold": 0.5,
    },
    timeout=20,
)
print("compare litsuggest", cmp.status_code, cmp.json() if cmp.headers.get("content-type", "").startswith("application/json") else cmp.text[:220])

# Optional PubMed check (set RUN_PUBMED_CHECK=True and provide a real email)
RUN_PUBMED_CHECK = False
PUBMED_EMAIL = "you@institution.edu"
if RUN_PUBMED_CHECK:
    if "@" not in PUBMED_EMAIL:
        raise ValueError("Set PUBMED_EMAIL before running PubMed fetch check")
    pf = requests.post(
        f"{base}/projects/{project_id}/data/pubmed/fetch",
        json={
            "email": PUBMED_EMAIL,
            "query": '"diabetic kidney disease"[Title/Abstract]',
            "max_results": 5,
            "min_abstract_chars": 0,
        },
        timeout=80,
    )
    print("pubmed/fetch", pf.status_code, pf.json() if pf.headers.get("content-type", "").startswith("application/json") else pf.text[:220])


## 9) Stop server and tunnel


In [ ]:
from pyngrok import ngrok

for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    print('Backend terminated.')

print('Cleanup done.')
